# Первичный просмотр исходных данных NCA

Notebook показывает один кадр из `NCA/data_v2` и считает, сколько кадров всего лежит в исходных `.npy` последовательностях.

In [23]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "NCA" / "data_v2").exists():
            return candidate
    raise FileNotFoundError("Не найдена папка NCA/data_v2 относительно текущего notebook.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "NCA" / "data_v2"
DATA_ROOT

WindowsPath('D:/Proga/Game_of_life/Real_game_of_life/NCA/data_v2')

## Сколько кадров всего

In [24]:
files = sorted(DATA_ROOT.glob("*/*.npy"))
if not files:
    raise FileNotFoundError(f"В {DATA_ROOT} не найдено .npy файлов в подпапках split-ов.")

rows = []
for path in files:
    array = np.load(path, mmap_mode="r")
    if array.ndim != 4:
        raise ValueError(f"{path} должен иметь форму [T, H, W, C], получено {array.shape}")
    rows.append(
        {
            "split": path.parent.name,
            "file": path.name,
            "frames": int(array.shape[0]),
            "height": int(array.shape[1]),
            "width": int(array.shape[2]),
            "channels": int(array.shape[3]),
        }
    )

metadata = pd.DataFrame(rows)
total_frames = int(metadata["frames"].sum())

print(f"Файлов-последовательностей: {len(metadata)}")
print(f"Всего кадров: {total_frames}")
display(metadata.groupby("split", as_index=False).agg(files=("file", "count"), frames=("frames", "sum")))
display(metadata.head())

Файлов-последовательностей: 57
Всего кадров: 2394


,split,files,frames
0,test,10,420
1,train,40,1680
2,val,7,294


,split,file,frames,height,width,channels
0,test,HeLa-S3_nuc_pos18_q1_5c.npy,42,64,64,1
1,test,HeLa-S3_nuc_pos1_q0_5c.npy,42,64,64,1
2,test,HeLa-S3_nuc_pos1_q1_5c.npy,42,64,64,1
3,test,HeLa-S3_nuc_pos1_q2_5c.npy,42,64,64,1
4,test,HeLa-S3_nuc_pos1_q3_5c.npy,42,64,64,1


## Один кадр из данных

In [25]:
# Можно поменять split, номер файла, кадр и канал для просмотра других примеров.
split = "train"
file_index = 0
frame_index = 0
channel = 0

split_files = sorted((DATA_ROOT / split).glob("*.npy"))
if not split_files:
    raise FileNotFoundError(f"В split={split!r} не найдено .npy файлов.")

sample_path = split_files[file_index]
sample = np.load(sample_path, mmap_mode="r")
frame = np.asarray(sample[frame_index, :, :, channel])

print(f"Файл: {sample_path.relative_to(PROJECT_ROOT)}")
print(f"Форма последовательности: {sample.shape}")
print(f"Кадр: {frame_index}, канал: {channel}")
print(f"min={frame.min():.4f}, max={frame.max():.4f}, mean={frame.mean():.4f}, nonzero={(frame > 0).mean():.2%}")

fig = px.imshow(
    frame,
    color_continuous_scale="magma",
    title=f"{sample_path.stem} | frame {frame_index}",
    labels=dict(color="density"),
    origin="upper",
)
fig.update_layout(
    width=600, height=600,
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
)
fig.show()

Файл: NCA\data_v2\train\HeLa-S3_nuc_pos0_q0_5c.npy
Форма последовательности: (42, 64, 64, 1)
Кадр: 0, канал: 0
min=0.0000, max=1.0000, mean=0.0473, nonzero=8.28%


## GNN-граф для выбранного кадра

Эта ячейка берёт выбранные выше `sample_path` и `frame_index`, находит соответствующую последовательность в GNN-таблице спотов и строит граф тем же проектным кодом, который используется для GNN.

In [26]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from GNN.graph_dataset import FrameGraphDatasetConfig, build_frame_graphs, load_processed_spots


gnn_cfg = FrameGraphDatasetConfig()
spots = load_processed_spots(gnn_cfg.source_path)

sample_name = sample_path.stem
sequence_col = gnn_cfg.sequence_col
frame_col = gnn_cfg.frame_col

sequence_mask = spots[sequence_col].astype(str).str.endswith(sample_name)
if "sequence_name" in spots.columns:
    sequence_mask |= spots["sequence_name"].astype(str).eq(sample_name)

sequence_ids = sorted(spots.loc[sequence_mask, sequence_col].astype(str).unique())
if not sequence_ids:
    raise ValueError(f"Не нашлась GNN-последовательность для {sample_name!r}.")

sequence_uid = sequence_ids[0]
sequence_spots = spots.loc[spots[sequence_col].astype(str).eq(sequence_uid)].copy()
frame_graphs = build_frame_graphs(sequence_spots, gnn_cfg, add_targets=False)

matching_graphs = [
    graph
    for graph in frame_graphs
    if not graph.nodes.empty and int(graph.nodes[frame_col].iloc[0]) == int(frame_index)
]
if not matching_graphs:
    available = sorted(sequence_spots[frame_col].dropna().astype(int).unique().tolist())
    raise ValueError(f"Для {sequence_uid!r} нет GNN-графа frame={frame_index}. Доступные кадры: {available[:10]}...")

graph = matching_graphs[0]
nodes = graph.nodes.reset_index(drop=True)
edges = graph.edges.reset_index(drop=True)
pos = nodes.loc[:, list(gnn_cfg.position_cols)].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)


def parse_local_contour(value):
    if not isinstance(value, str) or not value:
        return None
    points = []
    for pair in value.split("|"):
        if ":" not in pair:
            continue
        x_text, y_text = pair.split(":", 1)
        try:
            points.append((float(x_text), float(y_text)))
        except ValueError:
            continue
    return np.asarray(points, dtype=float) if len(points) >= 3 else None


print(f"NCA-файл: {sample_path.relative_to(PROJECT_ROOT)}")
print(f"GNN sequence_uid: {sequence_uid}")
print(f"GNN frame: {frame_index}")
print(f"Узлов: {graph.num_nodes}")
print(f"Рёбер: {graph.num_edges}")
print(f"Радиус рёбер: {gnn_cfg.edge_radius}, kNN: {gnn_cfg.edge_k_nearest}, bidirectional: {gnn_cfg.bidirectional_edges}")

fig = go.Figure()

# Edges as one trace with None separators
if graph.num_edges:
    edge_x, edge_y = [], []
    for source, target in zip(graph.edge_index[0], graph.edge_index[1]):
        edge_x += [float(pos[source][0]), float(pos[target][0]), None]
        edge_y += [float(pos[source][1]), float(pos[target][1]), None]
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(color="#4f8cff", width=0.8),
        opacity=0.35, showlegend=False, hoverinfo="skip",
    ))

# Cell contours
contours_drawn = 0
if "contour_xy_local" in nodes.columns:
    for index, row in nodes.iterrows():
        contour = parse_local_contour(row["contour_xy_local"])
        if contour is None:
            continue
        contour_xy = contour + pos[index]
        cx = np.append(contour_xy[:, 0], contour_xy[0, 0]).tolist()
        cy = np.append(contour_xy[:, 1], contour_xy[0, 1]).tolist()
        fig.add_trace(go.Scatter(
            x=cx, y=cy, mode="lines", fill="toself",
            fillcolor="rgba(242,201,76,0.28)",
            line=dict(color="#8a5a00", width=0.9),
            showlegend=False, hoverinfo="skip",
        ))
        contours_drawn += 1

# Node sizes
if "AREA" in nodes.columns:
    area = pd.to_numeric(nodes["AREA"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    node_sizes = (6 + 20 * np.sqrt(area / max(float(np.nanmax(area)), 1.0))).tolist()
elif "RADIUS" in nodes.columns:
    radius_vals = pd.to_numeric(nodes["RADIUS"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    node_sizes = (6 + 20 * radius_vals / max(float(np.nanmax(radius_vals)), 1.0)).tolist()
else:
    node_sizes = 8

node_id_vals = nodes[gnn_cfg.node_id_col].tolist()

fig.add_trace(go.Scatter(
    x=pos[:, 0].tolist(), y=pos[:, 1].tolist(),
    mode="markers+text",
    marker=dict(color="#d94f45", size=node_sizes, line=dict(color="white", width=0.5)),
    text=[str(v) for v in node_id_vals],
    textposition="top right", textfont=dict(size=7),
    opacity=0.9, name="cells",
))

fig.update_layout(
    title=f"GNN-граф: {sample_name}, frame {frame_index}",
    xaxis_title="x", yaxis_title="y",
    yaxis=dict(autorange="reversed", scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain="domain"),
    width=700, height=700,
    plot_bgcolor="white",
)
fig.show()
print(f"Контуров клеток показано: {contours_drawn}")

NCA-файл: NCA\data_v2\train\HeLa-S3_nuc_pos0_q0_5c.npy
GNN sequence_uid: seq0006_HeLa-S3_nuc_pos0_q0_5c
GNN frame: 0
Узлов: 22
Рёбер: 42
Радиус рёбер: 80.0, kNN: 0, bidirectional: True


Контуров клеток показано: 22


## Локальная область предсказания вокруг клетки 6972

Здесь `spot_id=6972` ставится в центр координат `(0, 0)`. Рисуется k-hop окрестность по GNN-графу: при `receptive_hops = 4` это соответствует стандартному числу message passing слоёв в one-step GNN.

In [27]:
center_spot_id = 6972
receptive_hops = 2
show_contours = True
show_all_direct_radius = True

if "graph" not in globals() or "nodes" not in globals() or "pos" not in globals():
    raise RuntimeError("Сначала выполните предыдущую ячейку, которая строит GNN-граф для выбранного кадра.")

node_ids_numeric = pd.to_numeric(nodes[gnn_cfg.node_id_col], errors="coerce")
center_matches = np.flatnonzero(node_ids_numeric.to_numpy() == center_spot_id)
if len(center_matches) == 0:
    available_ids = node_ids_numeric.dropna().astype(int).tolist()
    raise ValueError(
        f"spot_id={center_spot_id} не найден в текущем графе frame={frame_index}. "
        f"Первые доступные spot_id: {available_ids[:20]}"
    )
center_idx = int(center_matches[0])
center_xy = pos[center_idx].copy()

adjacency = {index: set() for index in range(graph.num_nodes)}
for source, target in graph.edge_index.T:
    source = int(source)
    target = int(target)
    adjacency[source].add(target)
    adjacency[target].add(source)

hop_by_node = {center_idx: 0}
frontier = {center_idx}
for hop in range(1, receptive_hops + 1):
    next_frontier = set()
    for node in frontier:
        next_frontier.update(adjacency[node])
    next_frontier -= set(hop_by_node)
    for node in sorted(next_frontier):
        hop_by_node[node] = hop
    frontier = next_frontier

selected = sorted(hop_by_node, key=lambda index: (hop_by_node[index], index))
selected_set = set(selected)
relative_pos = pos - center_xy

selected_edges = []
for source, target in graph.edge_index.T:
    source = int(source)
    target = int(target)
    if source in selected_set and target in selected_set:
        selected_edges.append((source, target))

print(f"Центральная клетка spot_id: {center_spot_id}")
print(f"Абсолютные координаты центра: x={center_xy[0]:.2f}, y={center_xy[1]:.2f}")
print(f"receptive_hops: {receptive_hops}")
print(f"Узлов в области: {len(selected)} из {graph.num_nodes}")
print(f"Рёбер внутри области: {len(selected_edges)}")
print("Узлов по hop:", {hop: sum(1 for value in hop_by_node.values() if value == hop) for hop in range(receptive_hops + 1)})

hop_colors = {0: "#d7191c", 1: "#fdae61", 2: "#2b83ba", 3: "#abdda4", 4: "#8073ac"}

fig = go.Figure()

# Radius circle
if show_all_direct_radius and gnn_cfg.edge_radius is not None:
    theta = np.linspace(0, 2 * np.pi, 200)
    r = float(gnn_cfg.edge_radius)
    fig.add_trace(go.Scatter(
        x=(r * np.cos(theta)).tolist(), y=(r * np.sin(theta)).tolist(),
        mode="lines",
        line=dict(color="#555555", width=1.0, dash="dash"),
        opacity=0.55, showlegend=True, hoverinfo="skip",
        name=f"radius={gnn_cfg.edge_radius:g}",
    ))

# Edges
if selected_edges:
    edge_x, edge_y = [], []
    for source, target in selected_edges:
        edge_x += [float(relative_pos[source][0]), float(relative_pos[target][0]), None]
        edge_y += [float(relative_pos[source][1]), float(relative_pos[target][1]), None]
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(color="#4f8cff", width=0.8),
        opacity=0.35, showlegend=False, hoverinfo="skip",
    ))

# Cell contours
if show_contours and "contour_xy_local" in nodes.columns:
    for index in selected:
        contour = parse_local_contour(nodes.loc[index, "contour_xy_local"])
        if contour is None:
            continue
        is_center = index == center_idx
        contour_xy = contour + relative_pos[index]
        cx = np.append(contour_xy[:, 0], contour_xy[0, 0]).tolist()
        cy = np.append(contour_xy[:, 1], contour_xy[0, 1]).tolist()
        fig.add_trace(go.Scatter(
            x=cx, y=cy, mode="lines", fill="toself",
            fillcolor="rgba(228,87,46,0.45)" if is_center else "rgba(242,201,76,0.25)",
            line=dict(color="#8b0000" if is_center else "#8a5a00", width=1.8 if is_center else 0.9),
            showlegend=False, hoverinfo="skip",
        ))

# Nodes grouped by hop
area_max_val = float(np.nanmax(pd.to_numeric(nodes["AREA"], errors="coerce").fillna(0))) if "AREA" in nodes.columns else 1.0
for hop in range(receptive_hops + 1):
    hop_nodes = [index for index in selected if hop_by_node[index] == hop]
    if not hop_nodes:
        continue
    xy = relative_pos[hop_nodes]
    if "AREA" in nodes.columns:
        area = pd.to_numeric(nodes.loc[hop_nodes, "AREA"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        sizes = (8 + 20 * np.sqrt(area / max(area_max_val, 1.0))).tolist()
    else:
        sizes = 12 if hop == 0 else 8
    hop_ids = [
        str(int(node_ids_numeric.iloc[index])) if pd.notna(node_ids_numeric.iloc[index]) else ""
        for index in hop_nodes
    ]
    fig.add_trace(go.Scatter(
        x=xy[:, 0].tolist(), y=xy[:, 1].tolist(),
        mode="markers+text",
        marker=dict(color=hop_colors.get(hop, "#999999"), size=sizes, line=dict(color="white", width=0.7)),
        text=hop_ids,
        textposition="top right", textfont=dict(size=7),
        opacity=0.95,
        name="центр" if hop == 0 else f"{hop}-hop",
    ))

fig.add_hline(y=0, line=dict(color="#222222", width=0.8), opacity=0.35)
fig.add_vline(x=0, line=dict(color="#222222", width=0.8), opacity=0.35)

fig.update_layout(
    title=f"Область GNN-предсказания вокруг spot_id={center_spot_id}",
    xaxis_title="x относительно 6972",
    yaxis_title="y относительно 6972",
    yaxis=dict(autorange="reversed", scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain="domain"),
    width=700, height=700,
    plot_bgcolor="white",
)
fig.show()

Центральная клетка spot_id: 6972
Абсолютные координаты центра: x=239.57, y=283.05
receptive_hops: 2
Узлов в области: 9 из 22
Рёбер внутри области: 22
Узлов по hop: {0: 1, 1: 4, 2: 4}


## NCA-область вокруг клетки 6972

Эта визуализация переносит `spot_id=6972` на NCA-сетку `64x64`: показывает чёткие границы grid-ячеек, контуры клеток, размеры клеток и фиксированную локальную область одного NCA update вокруг центра.

In [28]:
center_spot_id = 6972
nca_kernel_size = 3
nca_conv_layers_per_step = 2
nca_steps_to_show = [1]

if "frame" not in globals() or "sample_path" not in globals():
    raise RuntimeError("Сначала выполните ячейку, которая выбирает и показывает NCA frame.")
if "nodes" not in globals() or "pos" not in globals():
    raise RuntimeError("Сначала выполните ячейку, которая связывает выбранный кадр с GNN-таблицей спотов.")
if "parse_local_contour" not in globals():
    raise RuntimeError("Сначала выполните ячейку визуализации GNN-графа: там определяется parse_local_contour().")

node_ids_numeric = pd.to_numeric(nodes[gnn_cfg.node_id_col], errors="coerce")
center_matches = np.flatnonzero(node_ids_numeric.to_numpy() == center_spot_id)
if len(center_matches) == 0:
    available_ids = node_ids_numeric.dropna().astype(int).tolist()
    raise ValueError(f"spot_id={center_spot_id} не найден. Первые доступные spot_id: {available_ids[:20]}")

center_idx = int(center_matches[0])
center_xy_raw = pos[center_idx].copy()

qc_row = pd.read_csv(DATA_ROOT / "qc.csv")
qc_match = qc_row[qc_row["name"].astype(str).eq(sample_path.stem)]
if not qc_match.empty and "grid_size" in qc_match.columns:
    grid_size = float(qc_match.iloc[0]["grid_size"])
else:
    grid_size = 8.0

center_xy_grid = center_xy_raw / grid_size
frame_height, frame_width = frame.shape
one_step_radius = (nca_kernel_size // 2) * nca_conv_layers_per_step

print(f"NCA frame: {sample_path.relative_to(PROJECT_ROOT)}, frame_index={frame_index}")
print(f"Центр: spot_id={center_spot_id}")
print(f"Абсолютные координаты TrackMate: x={center_xy_raw[0]:.2f}, y={center_xy_raw[1]:.2f}")
print(f"Координаты на NCA grid: x={center_xy_grid[0]:.2f}, y={center_xy_grid[1]:.2f}")
print(f"grid_size: {grid_size:g} raw pixels per NCA cell")
print(f"Эффективный радиус одного NCA update: {one_step_radius} grid cells")

fig = go.Figure()

# Minor grid at every cell boundary (offset 0.5)
fig.update_xaxes(dtick=1, tick0=-0.5, showgrid=True, gridcolor="#b8b8b8", gridwidth=0.55)
fig.update_yaxes(dtick=1, tick0=-0.5, showgrid=True, gridcolor="#b8b8b8", gridwidth=0.55)

# Major grid every 5 cells
for gx in np.arange(-0.5, frame_width + 0.5, 5):
    fig.add_vline(x=float(gx), line=dict(color="#777777", width=0.85), opacity=0.65)
for gy in np.arange(-0.5, frame_height + 0.5, 5):
    fig.add_hline(y=float(gy), line=dict(color="#777777", width=0.85), opacity=0.65)

# NCA update rectangles
area_colors = ["#fdae61", "#2b83ba", "#8073ac", "#1a9850"]
for color, steps in zip(area_colors, nca_steps_to_show):
    radius = one_step_radius * int(steps)
    x0 = float(center_xy_grid[0] - radius - 0.5)
    y0 = float(center_xy_grid[1] - radius - 0.5)
    width = 2 * radius + 1
    fig.add_shape(type="rect",
        x0=x0, y0=y0, x1=x0 + width, y1=y0 + width,
        line=dict(color=color, width=1.6),
        fillcolor="rgba(0,0,0,0)",
    )
    fig.add_annotation(
        x=x0, y=y0 - 0.8,
        text=f"one NCA update: {width:g}x{width:g}",
        showarrow=False, font=dict(color=color, size=9),
        yanchor="bottom", xanchor="left",
    )

# Cell contours in NCA grid coordinates
visible_cells = 0
for index, row in nodes.iterrows():
    contour = parse_local_contour(row.get("contour_xy_local")) if "contour_xy_local" in nodes.columns else None
    if contour is None:
        continue
    cell_center_grid = pos[index] / grid_size
    contour_grid = contour / grid_size + cell_center_grid
    is_center = index == center_idx
    cx = np.append(contour_grid[:, 0], contour_grid[0, 0]).tolist()
    cy = np.append(contour_grid[:, 1], contour_grid[0, 1]).tolist()
    fig.add_trace(go.Scatter(
        x=cx, y=cy, mode="lines", fill="toself",
        fillcolor="rgba(228,87,46,0.50)" if is_center else "rgba(242,201,76,0.24)",
        line=dict(color="#8b0000" if is_center else "#8a5a00", width=2.0 if is_center else 0.9),
        showlegend=False, hoverinfo="skip",
    ))
    visible_cells += 1

# Cell centers with labels
area_values = pd.to_numeric(nodes["AREA"], errors="coerce") if "AREA" in nodes.columns else pd.Series(np.nan, index=nodes.index)
radius_values = pd.to_numeric(nodes["RADIUS"], errors="coerce") if "RADIUS" in nodes.columns else pd.Series(np.nan, index=nodes.index)
near_radius = max(one_step_radius * max(nca_steps_to_show), 8)

sc_x, sc_y, sc_colors, sc_sizes, sc_text = [], [], [], [], []
for index, row in nodes.iterrows():
    cell_center_grid = pos[index] / grid_size
    dist = float(np.linalg.norm(cell_center_grid - center_xy_grid))
    if dist > near_radius + 2 and index != center_idx:
        continue
    is_center = index == center_idx
    spot_id = int(node_ids_numeric.iloc[index]) if pd.notna(node_ids_numeric.iloc[index]) else row[gnn_cfg.node_id_col]
    area_raw = area_values.iloc[index]
    radius_raw = radius_values.iloc[index]
    label_parts = [str(spot_id)]
    if pd.notna(area_raw):
        label_parts.append(f"A={area_raw / (grid_size ** 2):.1f}")
    if pd.notna(radius_raw):
        label_parts.append(f"R={radius_raw / grid_size:.1f}")
    sc_x.append(float(cell_center_grid[0]))
    sc_y.append(float(cell_center_grid[1]))
    sc_colors.append("#d7191c" if is_center else "#2b83ba")
    sc_sizes.append(14 if is_center else 8)
    sc_text.append(" ".join(label_parts))

fig.add_trace(go.Scatter(
    x=sc_x, y=sc_y,
    mode="markers+text",
    marker=dict(color=sc_colors, size=sc_sizes, line=dict(color="white", width=0.7)),
    text=sc_text, textposition="top right", textfont=dict(size=7),
    showlegend=False,
))

fig.add_hline(y=float(center_xy_grid[1]), line=dict(color="#222222", width=0.8), opacity=0.28)
fig.add_vline(x=float(center_xy_grid[0]), line=dict(color="#222222", width=0.8), opacity=0.28)

xlim_left = float(max(center_xy_grid[0] - near_radius - 4, -0.5))
xlim_right = float(min(center_xy_grid[0] + near_radius + 4, frame_width - 0.5))
ylim_bottom = float(min(center_xy_grid[1] + near_radius + 4, frame_height - 0.5))
ylim_top = float(max(center_xy_grid[1] - near_radius - 4, -0.5))

fig.update_layout(
    title=f"NCA grid вокруг spot_id={center_spot_id}: области update и контуры клеток",
    xaxis_title="NCA grid x",
    yaxis_title="NCA grid y",
    xaxis=dict(range=[xlim_left, xlim_right], constrain="domain"),
    yaxis=dict(range=[ylim_bottom, ylim_top], scaleanchor="x", scaleratio=1),
    width=800, height=800,
    plot_bgcolor="white",
)
fig.show()

print(f"Контуров клеток наложено: {visible_cells}")
print("Прямоугольник показывает фиксированную локальную область одного NCA update вокруг клетки 6972.")
print("Контуры и размеры клеток взяты из TrackMate/GNN таблицы, но переведены в координаты NCA grid.")

NCA frame: NCA\data_v2\train\HeLa-S3_nuc_pos0_q0_5c.npy, frame_index=0
Центр: spot_id=6972
Абсолютные координаты TrackMate: x=239.57, y=283.05
Координаты на NCA grid: x=29.95, y=35.38
grid_size: 8 raw pixels per NCA cell
Эффективный радиус одного NCA update: 2 grid cells


Контуров клеток наложено: 22
Прямоугольник показывает фиксированную локальную область одного NCA update вокруг клетки 6972.
Контуры и размеры клеток взяты из TrackMate/GNN таблицы, но переведены в координаты NCA grid.


## NCA + GNN вокруг клетки 6972 на одном графике

Здесь фиксированная NCA update-область, границы NCA grid-ячеек и GNN-граф совмещены в координатах NCA grid. Центр — `spot_id=6972`.

In [29]:
center_spot_id = 6972
combined_gnn_hops = globals().get("receptive_hops", 2)
nca_steps_to_show = [1]
nca_kernel_size = 3
nca_conv_layers_per_step = 2

if "frame" not in globals() or "sample_path" not in globals():
    raise RuntimeError("Сначала выполните ячейку, которая выбирает NCA frame.")
if "graph" not in globals() or "nodes" not in globals() or "pos" not in globals():
    raise RuntimeError("Сначала выполните ячейку, которая строит GNN-граф для выбранного кадра.")
if "parse_local_contour" not in globals():
    raise RuntimeError("Сначала выполните ячейку визуализации GNN-графа: там определяется parse_local_contour().")

node_ids_numeric = pd.to_numeric(nodes[gnn_cfg.node_id_col], errors="coerce")
center_matches = np.flatnonzero(node_ids_numeric.to_numpy() == center_spot_id)
if len(center_matches) == 0:
    available_ids = node_ids_numeric.dropna().astype(int).tolist()
    raise ValueError(f"spot_id={center_spot_id} не найден. Первые доступные spot_id: {available_ids[:20]}")
center_idx = int(center_matches[0])

qc = pd.read_csv(DATA_ROOT / "qc.csv")
qc_match = qc[qc["name"].astype(str).eq(sample_path.stem)]
grid_size = float(qc_match.iloc[0]["grid_size"]) if not qc_match.empty and "grid_size" in qc_match.columns else 8.0
pos_grid = pos / grid_size
center_grid = pos_grid[center_idx].copy()
frame_height, frame_width = frame.shape
one_step_radius = (nca_kernel_size // 2) * nca_conv_layers_per_step

# GNN k-hop neighbourhood
adjacency = {index: set() for index in range(graph.num_nodes)}
for source, target in graph.edge_index.T:
    adjacency[int(source)].add(int(target))
    adjacency[int(target)].add(int(source))

hop_by_node = {center_idx: 0}
frontier = {center_idx}
for hop in range(1, combined_gnn_hops + 1):
    next_frontier = set()
    for node in frontier:
        next_frontier.update(adjacency[node])
    next_frontier -= set(hop_by_node)
    for node in sorted(next_frontier):
        hop_by_node[node] = hop
    frontier = next_frontier
selected = sorted(hop_by_node, key=lambda index: (hop_by_node[index], index))
selected_set = set(selected)

selected_edges = [
    (int(s), int(t))
    for s, t in graph.edge_index.T
    if int(s) in selected_set and int(t) in selected_set
]

fig = go.Figure()

# Minor grid at every NCA cell boundary
fig.update_xaxes(dtick=1, tick0=-0.5, showgrid=True, gridcolor="#b8b8b8", gridwidth=0.55)
fig.update_yaxes(dtick=1, tick0=-0.5, showgrid=True, gridcolor="#b8b8b8", gridwidth=0.55)

# Major grid every 5
for gx in np.arange(-0.5, frame_width + 0.5, 5):
    fig.add_vline(x=float(gx), line=dict(color="#777777", width=0.85), opacity=0.65)
for gy in np.arange(-0.5, frame_height + 0.5, 5):
    fig.add_hline(y=float(gy), line=dict(color="#777777", width=0.85), opacity=0.65)

# NCA update rectangle
nca_rect_colors = {1: "#fdae61", 2: "#2b83ba", 4: "#8073ac"}
for steps in nca_steps_to_show:
    radius = one_step_radius * int(steps)
    width = 2 * radius + 1
    x0 = float(center_grid[0] - radius - 0.5)
    y0 = float(center_grid[1] - radius - 0.5)
    fig.add_shape(type="rect",
        x0=x0, y0=y0, x1=x0 + width, y1=y0 + width,
        line=dict(color=nca_rect_colors.get(steps, "#555555"), width=1.7),
        fillcolor="rgba(0,0,0,0)",
        name=f"NCA one update: {width:g}x{width:g}",
    )

# GNN edges in grid coordinates
if selected_edges:
    edge_x, edge_y = [], []
    for source, target in selected_edges:
        edge_x += [float(pos_grid[source][0]), float(pos_grid[target][0]), None]
        edge_y += [float(pos_grid[source][1]), float(pos_grid[target][1]), None]
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(color="#00a6ff", width=1.05),
        opacity=0.48, showlegend=True, hoverinfo="skip",
        name="GNN edges",
    ))

# GNN radius circle in grid units
if gnn_cfg.edge_radius is not None:
    gnn_radius_grid = float(gnn_cfg.edge_radius) / grid_size
    theta = np.linspace(0, 2 * np.pi, 200)
    fig.add_trace(go.Scatter(
        x=(center_grid[0] + gnn_radius_grid * np.cos(theta)).tolist(),
        y=(center_grid[1] + gnn_radius_grid * np.sin(theta)).tolist(),
        mode="lines",
        line=dict(color="#00a6ff", width=1.2, dash="dash"),
        opacity=0.75, showlegend=True, hoverinfo="skip",
        name=f"GNN radius={gnn_radius_grid:.1f} grid",
    ))

# Cell contours in grid coordinates
hop_colors = {0: "#d7191c", 1: "#fdae61", 2: "#2b83ba", 3: "#abdda4", 4: "#8073ac"}
area_values = pd.to_numeric(nodes["AREA"], errors="coerce") if "AREA" in nodes.columns else pd.Series(np.nan, index=nodes.index)
radius_values = pd.to_numeric(nodes["RADIUS"], errors="coerce") if "RADIUS" in nodes.columns else pd.Series(np.nan, index=nodes.index)
area_max_val = float(np.nanmax(area_values.dropna())) if area_values.notna().any() else 1.0

for index in selected:
    is_center = index == center_idx
    hop = hop_by_node[index]
    contour = parse_local_contour(nodes.loc[index, "contour_xy_local"]) if "contour_xy_local" in nodes.columns else None
    if contour is not None:
        contour_grid = contour / grid_size + pos_grid[index]
        cx = np.append(contour_grid[:, 0], contour_grid[0, 0]).tolist()
        cy = np.append(contour_grid[:, 1], contour_grid[0, 1]).tolist()
        fig.add_trace(go.Scatter(
            x=cx, y=cy, mode="lines", fill="toself",
            fillcolor="rgba(228,87,46,0.48)" if is_center else "rgba(242,201,76,0.30)",
            line=dict(color="#8b0000" if is_center else hop_colors.get(hop, "#999999"), width=2.0 if is_center else 1.0),
            showlegend=False, hoverinfo="skip",
        ))

# Nodes by hop
for hop in range(combined_gnn_hops + 1):
    hop_nodes = [index for index in selected if hop_by_node[index] == hop]
    if not hop_nodes:
        continue
    xy = pos_grid[hop_nodes]
    if "AREA" in nodes.columns:
        area = area_values.iloc[hop_nodes].fillna(0.0).to_numpy(dtype=float)
        sizes = (8 + 20 * np.sqrt(area / max(area_max_val, 1.0))).tolist()
    else:
        sizes = 14 if hop == 0 else 8
    labels = []
    for index in hop_nodes:
        a = area_values.iloc[index]
        r = radius_values.iloc[index]
        parts = [str(int(node_ids_numeric.iloc[index])) if pd.notna(node_ids_numeric.iloc[index]) else "?", f"h{hop}"]
        if pd.notna(a):
            parts.append(f"A={a / (grid_size ** 2):.1f}")
        if pd.notna(r):
            parts.append(f"R={r / grid_size:.1f}")
        labels.append(" ".join(parts))
    fig.add_trace(go.Scatter(
        x=xy[:, 0].tolist(), y=xy[:, 1].tolist(),
        mode="markers+text",
        marker=dict(color=hop_colors.get(hop, "#999999"), size=sizes, line=dict(color="white", width=0.8)),
        text=labels,
        textposition="top right", textfont=dict(size=7),
        opacity=0.95,
        name="центр GNN" if hop == 0 else f"{hop}-hop",
    ))

fig.add_hline(y=float(center_grid[1]), line=dict(color="#222222", width=0.8), opacity=0.28)
fig.add_vline(x=float(center_grid[0]), line=dict(color="#222222", width=0.8), opacity=0.28)

view_radius = max(
    one_step_radius * max(nca_steps_to_show),
    float(gnn_cfg.edge_radius or 0.0) / grid_size,
    8.0,
) + 4
xlim_left = float(max(center_grid[0] - view_radius, -0.5))
xlim_right = float(min(center_grid[0] + view_radius, frame_width - 0.5))
ylim_bottom = float(min(center_grid[1] + view_radius, frame_height - 0.5))
ylim_top = float(max(center_grid[1] - view_radius, -0.5))

fig.update_layout(
    title=f"NCA + GNN вокруг spot_id={center_spot_id}",
    xaxis_title="NCA grid x",
    yaxis_title="NCA grid y",
    xaxis=dict(range=[xlim_left, xlim_right], constrain="domain"),
    yaxis=dict(range=[ylim_bottom, ylim_top], scaleanchor="x", scaleratio=1),
    width=850, height=850,
    plot_bgcolor="white",
    legend=dict(x=1.01, y=1, xanchor="left", font=dict(size=8)),
)
fig.show()

print(f"Центр 6972 на NCA grid: x={center_grid[0]:.2f}, y={center_grid[1]:.2f}")
print(f"NCA: каждый update использует одно и то же эффективное окно {(2 * one_step_radius + 1)}x{(2 * one_step_radius + 1)} вокруг каждой grid-ячейки.")
print(f"GNN: показана {combined_gnn_hops}-hop область вокруг 6972: {len(selected)} узлов, {len(selected_edges)} направленных рёбер внутри области.")
print(f"Масштаб: 1 NCA grid cell = {grid_size:g} raw pixels TrackMate.")

Центр 6972 на NCA grid: x=29.95, y=35.38
NCA: каждый update использует одно и то же эффективное окно 5x5 вокруг каждой grid-ячейки.
GNN: показана 2-hop область вокруг 6972: 9 узлов, 22 направленных рёбер внутри области.
Масштаб: 1 NCA grid cell = 8 raw pixels TrackMate.


## One-step GNN: предсказание против реальности

Эта ячейка загружает сохранённый GNN checkpoint, делает одно предсказание для `spot_id=6972` на выбранном кадре и рисует текущую клетку, предсказанное следующее положение и реальное следующее положение.

In [30]:
from dataclasses import fields
import torch

from GNN.gnn_model import CellInteractionGNN
from GNN.graph_dataset import cell_graph_to_pyg_training_data

one_step_checkpoint_path = PROJECT_ROOT / "GNN" / "runs" / "gnn_baseline_local_5000_balanced" / "best.pt"
one_step_spot_id = 6972

if not one_step_checkpoint_path.exists():
    raise FileNotFoundError(f"Не найден GNN checkpoint: {one_step_checkpoint_path}")
if "parse_local_contour" not in globals():
    raise RuntimeError("Сначала выполните ячейку визуализации GNN-графа: там определяется parse_local_contour().")

checkpoint = torch.load(one_step_checkpoint_path, map_location="cpu", weights_only=False)
train_cfg = checkpoint["train_config"]
dataset_cfg_raw = checkpoint.get("dataset_config", {})
valid_cfg_fields = {field.name for field in fields(FrameGraphDatasetConfig)}
dataset_kwargs = {key: value for key, value in dataset_cfg_raw.items() if key in valid_cfg_fields}
if "source_path" in dataset_kwargs:
    dataset_kwargs["source_path"] = Path(dataset_kwargs["source_path"])
for key in (
    "position_cols",
    "horizons",
    "node_feature_columns",
    "extra_node_feature_columns",
    "edge_feature_columns",
    "temporal_lags",
):
    if key in dataset_kwargs and dataset_kwargs[key] is not None:
        dataset_kwargs[key] = tuple(dataset_kwargs[key])

prediction_cfg = FrameGraphDatasetConfig(**dataset_kwargs)
prediction_spots = load_processed_spots(prediction_cfg.source_path)

prediction_sequence_mask = prediction_spots[prediction_cfg.sequence_col].astype(str).str.endswith(sample_path.stem)
if "sequence_name" in prediction_spots.columns:
    prediction_sequence_mask |= prediction_spots["sequence_name"].astype(str).eq(sample_path.stem)
prediction_sequence_ids = sorted(prediction_spots.loc[prediction_sequence_mask, prediction_cfg.sequence_col].astype(str).unique())
if not prediction_sequence_ids:
    raise ValueError(f"Не найдена GNN-последовательность для {sample_path.stem!r}")

prediction_sequence_uid = prediction_sequence_ids[0]
prediction_sequence_spots = prediction_spots.loc[
    prediction_spots[prediction_cfg.sequence_col].astype(str).eq(prediction_sequence_uid)
].copy()
prediction_graphs = build_frame_graphs(prediction_sequence_spots, prediction_cfg, add_targets=True)
prediction_cell_graph = [
    item
    for item in prediction_graphs
    if not item.nodes.empty and int(item.nodes[prediction_cfg.frame_col].iloc[0]) == int(frame_index)
][0]
prediction_data = cell_graph_to_pyg_training_data(prediction_cell_graph, prediction_cfg)
prediction_nodes = prediction_cell_graph.nodes.reset_index(drop=True)
prediction_pos = prediction_nodes.loc[:, list(prediction_cfg.position_cols)].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

prediction_node_ids = pd.to_numeric(prediction_nodes[prediction_cfg.node_id_col], errors="coerce")
focus_matches = np.flatnonzero(prediction_node_ids.to_numpy() == one_step_spot_id)
if len(focus_matches) == 0:
    available_ids = prediction_node_ids.dropna().astype(int).tolist()
    raise ValueError(f"spot_id={one_step_spot_id} не найден в текущем кадре. Первые доступные: {available_ids[:20]}")
focus_index = int(focus_matches[0])

normalization = checkpoint.get("node_feature_normalization")
if normalization:
    mean = torch.as_tensor(normalization["mean"], dtype=torch.float32)
    std = torch.as_tensor(normalization["std"], dtype=torch.float32)
    prediction_data.x = (prediction_data.x.float() - mean) / std

num_division_horizons = sum(
    hasattr(prediction_data, f"target_division_within_{horizon}")
    for horizon in train_cfg.get("division_horizons", [3, 5, 10])
)
one_step_model = CellInteractionGNN(
    node_dim=int(prediction_data.x.size(-1)),
    edge_dim=int(prediction_data.edge_attr.size(-1)),
    shape_dim=int(prediction_data.target_delta_shape.size(-1)),
    hidden_dim=int(train_cfg["hidden_dim"]),
    num_message_passing_layers=int(train_cfg["layers"]),
    dropout=float(train_cfg.get("dropout", 0.0)),
    num_division_horizons=int(num_division_horizons),
)
one_step_model.load_state_dict(checkpoint["model_state"])
one_step_model.eval()

with torch.no_grad():
    one_step_output = one_step_model(prediction_data)

current_xy = prediction_pos[focus_index]
pred_delta = one_step_output.delta_pos[focus_index].detach().cpu().numpy()
real_delta = prediction_data.target_delta_pos[focus_index].detach().cpu().numpy()
predicted_xy = current_xy + pred_delta
real_xy = current_xy + real_delta
valid_real = bool(prediction_data.valid_regression_mask[focus_index].item())

division_prob = float(torch.sigmoid(one_step_output.division_logits[focus_index]).item())
death_prob = float(torch.sigmoid(one_step_output.death_logits[focus_index]).item())

qc = pd.read_csv(DATA_ROOT / "qc.csv")
qc_match = qc[qc["name"].astype(str).eq(sample_path.stem)]
grid_size = float(qc_match.iloc[0]["grid_size"]) if not qc_match.empty and "grid_size" in qc_match.columns else 8.0

def to_grid(xy):
    return np.asarray(xy, dtype=float) / grid_size

current_grid = to_grid(current_xy)
predicted_grid = to_grid(predicted_xy)
real_grid = to_grid(real_xy)

shape_target_columns = list(getattr(prediction_data, "shape_target_columns", []))
pred_shape_delta = one_step_output.delta_shape[focus_index].detach().cpu().numpy()
real_shape_delta = prediction_data.target_delta_shape[focus_index].detach().cpu().numpy()
valid_shape = bool(prediction_data.valid_shape_mask[focus_index].item()) if hasattr(prediction_data, "valid_shape_mask") else False

next_contour_grid = None
next_id = prediction_nodes.loc[focus_index, "next_id"] if "next_id" in prediction_nodes.columns else np.nan
if pd.notna(next_id):
    next_candidates = prediction_sequence_spots[
        (prediction_sequence_spots[prediction_cfg.node_id_col].astype(float) == float(next_id))
        & (prediction_sequence_spots[prediction_cfg.frame_col].astype(int) == int(frame_index) + 1)
    ]
    if not next_candidates.empty:
        next_row = next_candidates.iloc[0]
        next_contour = parse_local_contour(next_row.get("contour_xy_local"))
        if next_contour is not None:
            next_center = next_row.loc[list(prediction_cfg.position_cols)].astype(float).to_numpy()
            next_contour_grid = next_contour / grid_size + to_grid(next_center)

def sampled_contour_radii(contour_xy, sample_angles):
    if contour_xy is None or len(contour_xy) < 3:
        return None
    contour_xy = np.asarray(contour_xy, dtype=float)
    radii = np.linalg.norm(contour_xy, axis=1)
    angles = np.mod(np.arctan2(contour_xy[:, 1], contour_xy[:, 0]), 2.0 * np.pi)
    finite = np.isfinite(radii) & np.isfinite(angles)
    if finite.sum() < 3:
        return None
    angles = angles[finite]
    radii = radii[finite]
    order = np.argsort(angles)
    angles = angles[order]
    radii = radii[order]
    angles_ext = np.concatenate([angles - 2.0 * np.pi, angles, angles + 2.0 * np.pi])
    radii_ext = np.concatenate([radii, radii, radii])
    return np.interp(sample_angles, angles_ext, radii_ext)


def best_radial_phase_shift(current_contour, mean_radius, current_r_norm):
    sample_angles = np.linspace(0.0, 2.0 * np.pi, len(current_r_norm), endpoint=False)
    contour_radii = sampled_contour_radii(current_contour, sample_angles)
    if contour_radii is None:
        return 0
    base_radii = np.asarray(current_r_norm, dtype=float) * float(mean_radius)
    errors = [
        np.nanmean((np.roll(base_radii, shift) - contour_radii) ** 2)
        for shift in range(len(base_radii))
    ]
    if not np.isfinite(errors).any():
        return 0
    return int(np.nanargmin(errors))


def contour_from_predicted_radial_shape(row, center_grid, delta_shape, reference_contour=None):
    if not shape_target_columns or "shape_mean_radius" not in row.index:
        return None, 0
    try:
        mean_radius = float(row["shape_mean_radius"])
        current_r_norm = row.loc[shape_target_columns].astype(float).to_numpy(dtype=float)
        delta_shape = np.asarray(delta_shape, dtype=float)
    except (TypeError, ValueError):
        return None, 0
    if not np.isfinite(mean_radius) or mean_radius <= 0:
        return None, 0
    if current_r_norm.shape != delta_shape.shape or not np.isfinite(current_r_norm).all() or not np.isfinite(delta_shape).all():
        return None, 0

    phase_shift = best_radial_phase_shift(reference_contour, mean_radius, current_r_norm)
    predicted_r_norm = np.clip(current_r_norm + delta_shape, 0.05, None)
    predicted_r_norm = np.roll(predicted_r_norm, phase_shift)
    angles = np.linspace(0.0, 2.0 * np.pi, len(predicted_r_norm), endpoint=False)
    local_xy = np.column_stack([
        predicted_r_norm * mean_radius * np.cos(angles),
        predicted_r_norm * mean_radius * np.sin(angles),
    ])
    return local_xy / grid_size + center_grid, phase_shift


current_contour = parse_local_contour(prediction_nodes.loc[focus_index, "contour_xy_local"]) if "contour_xy_local" in prediction_nodes.columns else None
current_contour_grid = current_contour / grid_size + current_grid if current_contour is not None else None
predicted_contour_grid, predicted_shape_phase_shift = contour_from_predicted_radial_shape(
    prediction_nodes.loc[focus_index],
    predicted_grid,
    pred_shape_delta,
    reference_contour=current_contour,
)
if predicted_contour_grid is None and current_contour is not None:
    print("Не удалось восстановить предсказанный shape-контур; показываю текущий контур, перенесённый в predicted center.")
    predicted_contour_grid = current_contour / grid_size + predicted_grid

fig = go.Figure()

# Minor grid
fig.update_xaxes(dtick=1, tick0=-0.5, showgrid=True, gridcolor="#c7c7c7", gridwidth=0.5)
fig.update_yaxes(dtick=1, tick0=-0.5, showgrid=True, gridcolor="#c7c7c7", gridwidth=0.5)

# Local GNN edges touching the focus cell, kept faint so the three positions stay dominant.
for edge_source, edge_target in prediction_cell_graph.edge_index.T:
    edge_source = int(edge_source)
    edge_target = int(edge_target)
    if edge_source != focus_index and edge_target != focus_index:
        continue
    fig.add_trace(go.Scatter(
        x=[float(to_grid(prediction_pos[edge_source])[0]), float(to_grid(prediction_pos[edge_target])[0])],
        y=[float(to_grid(prediction_pos[edge_source])[1]), float(to_grid(prediction_pos[edge_target])[1])],
        mode="lines",
        line=dict(color="#8db7ff", width=0.8),
        opacity=0.18, showlegend=False, hoverinfo="skip",
    ))

# Cell contours
if current_contour_grid is not None:
    cx = np.append(current_contour_grid[:, 0], current_contour_grid[0, 0]).tolist()
    cy = np.append(current_contour_grid[:, 1], current_contour_grid[0, 1]).tolist()
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines", fill="toself",
        fillcolor="rgba(80,80,80,0.18)", line=dict(color="#3f3f3f", width=2.2),
        showlegend=True, name="контур: исходная"))

if predicted_contour_grid is not None:
    cx = np.append(predicted_contour_grid[:, 0], predicted_contour_grid[0, 0]).tolist()
    cy = np.append(predicted_contour_grid[:, 1], predicted_contour_grid[0, 1]).tolist()
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines", fill="toself",
        fillcolor="rgba(215,25,28,0.08)", line=dict(color="#d7191c", width=2.6, dash="dash"),
        showlegend=True, name="контур: GNN prediction"))

if next_contour_grid is not None:
    cx = np.append(next_contour_grid[:, 0], next_contour_grid[0, 0]).tolist()
    cy = np.append(next_contour_grid[:, 1], next_contour_grid[0, 1]).tolist()
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines", fill="toself",
        fillcolor="rgba(26,152,80,0.08)", line=dict(color="#1a9850", width=2.8),
        showlegend=True, name="контур: реальность t+1"))

# Movement vectors and error segment. Draw these before markers so labels stay readable.
fig.add_trace(go.Scatter(
    x=[float(current_grid[0]), float(predicted_grid[0])],
    y=[float(current_grid[1]), float(predicted_grid[1])],
    mode="lines",
    line=dict(color="#d7191c", width=3.0, dash="dash"),
    name="вектор: GNN prediction",
    hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=[float(current_grid[0]), float(real_grid[0])],
    y=[float(current_grid[1]), float(real_grid[1])],
    mode="lines",
    line=dict(color="#1a9850", width=3.2),
    name="вектор: реальность",
    hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=[float(predicted_grid[0]), float(real_grid[0])],
    y=[float(predicted_grid[1]), float(real_grid[1])],
    mode="lines+text",
    text=["", f"ошибка {np.linalg.norm(predicted_grid - real_grid):.2f}"],
    textposition="middle right",
    textfont=dict(size=13, color="#7a1f1f"),
    line=dict(color="#7a1f1f", width=2.2, dash="dot"),
    name="ошибка prediction vs reality",
))

# Node markers
marker_line = dict(color="white", width=1.8)
fig.add_trace(go.Scatter(
    x=[float(current_grid[0])], y=[float(current_grid[1])],
    mode="markers+text", text=[f"start {one_step_spot_id}"],
    textposition="top center", textfont=dict(size=13, color="#2f2f2f"),
    marker=dict(color="#3f3f3f", size=17, symbol="circle", line=marker_line),
    name="исходное положение t",
))
fig.add_trace(go.Scatter(
    x=[float(predicted_grid[0])], y=[float(predicted_grid[1])],
    mode="markers+text", text=["GNN"],
    textposition="top right", textfont=dict(size=14, color="#d7191c"),
    marker=dict(color="#d7191c", size=18, symbol="x", line=dict(color="#d7191c", width=2.2)),
    name="предсказанное t+1",
))
fig.add_trace(go.Scatter(
    x=[float(real_grid[0])], y=[float(real_grid[1])],
    mode="markers+text", text=["real"],
    textposition="bottom right", textfont=dict(size=14, color="#1a9850"),
    marker=dict(color="#1a9850", size=18, symbol="diamond", line=marker_line),
    name="реальное t+1",
))

# Arrows (prediction and reality vectors)
fig.add_annotation(
    x=float(predicted_grid[0]), y=float(predicted_grid[1]),
    ax=float(current_grid[0]), ay=float(current_grid[1]),
    xref="x", yref="y", axref="x", ayref="y",
    showarrow=True, arrowhead=3, arrowwidth=2, arrowsize=1.0,
    arrowcolor="#d7191c", opacity=0.95,
)
fig.add_annotation(
    x=float(real_grid[0]), y=float(real_grid[1]),
    ax=float(current_grid[0]), ay=float(current_grid[1]),
    xref="x", yref="y", axref="x", ayref="y",
    showarrow=True, arrowhead=3, arrowwidth=2, arrowsize=1.0,
    arrowcolor="#1a9850", opacity=0.95,
)

# View range: tight local zoom around start, prediction and real next position.
view_points = np.vstack([current_grid, predicted_grid, real_grid])
if current_contour_grid is not None:
    view_points = np.vstack([view_points, current_contour_grid])
if predicted_contour_grid is not None:
    view_points = np.vstack([view_points, predicted_contour_grid])
if next_contour_grid is not None:
    view_points = np.vstack([view_points, next_contour_grid])
view_min = np.nanmin(view_points, axis=0)
view_max = np.nanmax(view_points, axis=0)
view_center = (view_min + view_max) / 2.0
view_span = float(max(view_max[0] - view_min[0], view_max[1] - view_min[1]))
zoom_span = max(view_span + 1.6, 4.0)
max_x = float(frame.shape[1] - 0.5)
max_y = float(frame.shape[0] - 0.5)
xlim = [float(max(view_center[0] - zoom_span / 2.0, -0.5)), float(min(view_center[0] + zoom_span / 2.0, max_x))]
ylim = [float(min(view_center[1] + zoom_span / 2.0, max_y)), float(max(view_center[1] - zoom_span / 2.0, -0.5))]

fig.update_layout(
    title=f"One-step GNN для spot_id={one_step_spot_id}: start -> prediction vs real t+1",
    xaxis_title="NCA grid x",
    yaxis_title="NCA grid y",
    xaxis=dict(range=xlim, constrain="domain", zeroline=False, showline=True, linewidth=1, linecolor="#555555"),
    yaxis=dict(range=ylim, scaleanchor="x", scaleratio=1, zeroline=False, showline=True, linewidth=1, linecolor="#555555"),
    width=900, height=820,
    plot_bgcolor="white",
    font=dict(size=13),
    legend=dict(
        x=0.01, y=0.99, xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.86)", bordercolor="#dddddd", borderwidth=1,
        font=dict(size=11),
    ),
    margin=dict(l=70, r=30, t=70, b=65),
)
fig.show()

error_raw = np.linalg.norm(predicted_xy - real_xy)
error_grid = error_raw / grid_size
print(f"checkpoint: {one_step_checkpoint_path.relative_to(PROJECT_ROOT)}")
print(f"valid real next position: {valid_real}")
print(f"current xy: x={current_xy[0]:.3f}, y={current_xy[1]:.3f}")
print(f"predicted delta: dx={pred_delta[0]:.4f}, dy={pred_delta[1]:.4f}")
print(f"real delta: dx={real_delta[0]:.4f}, dy={real_delta[1]:.4f}")
shape_error = np.linalg.norm(pred_shape_delta - real_shape_delta)
print(f"position error: {error_raw:.4f} raw units = {error_grid:.4f} NCA grid cells")
print(f"valid shape target: {valid_shape}")
print(f"shape delta error: {shape_error:.4f} over {len(shape_target_columns)} radial shape features")
print(f"predicted shape phase shift: {predicted_shape_phase_shift} / {len(shape_target_columns)} radial bins")
print(f"division probability: {division_prob:.4f}")
print(f"death probability: {death_prob:.4f}")

checkpoint: GNN\runs\gnn_baseline_local_5000_balanced\best.pt
valid real next position: True
current xy: x=239.570, y=283.045
predicted delta: dx=-0.0006, dy=0.1177
real delta: dx=-0.0324, dy=0.0262
position error: 0.0969 raw units = 0.0121 NCA grid cells
valid shape target: True
shape delta error: 0.3085 over 64 radial shape features
predicted shape phase shift: 56 / 64 radial bins
division probability: 0.0086
death probability: 0.2216


## Сравнение rollout-моделей: baseline vs BPTT-3 vs BPTT-5

Ниже показаны метрики на **одном и том же test split** для трёх реально обученных моделей:

- **Baseline** — one-step GNN, обученная предсказывать только следующий кадр.
- **BPTT-3** — GNN, обученная дифференцируемым rollout на 3 шага с перестроением radius/kNN-рёбер после каждого шага.
- **BPTT-5** — та же схема, но loss распространяется через 5 шагов.

Метрики:

- `position mean` и `position RMSE` — ошибка положения клетки в исходных координатах датасета; меньше — лучше.
- `shape RMSE` — ошибка радиального профиля формы `shape_r_norm_*`; меньше — лучше.
- `valid shape fraction` — доля предсказанных форм с конечными радиусами в допустимом диапазоне `[0.05, 3.0]`; больше — лучше.

Число `matched_nodes` уменьшается с горизонтом, потому что не все GT-треки продолжаются до 20-го шага. Это одинаковый набор клеток для всех моделей на каждом горизонте и не является количеством клеток, которое модель «сохранила».


In [32]:
import json

rollout_eval_path = PROJECT_ROOT / "GNN" / "runs" / "rollout_eval_baseline_vs_bptt3_vs_bptt5.json"
if not rollout_eval_path.exists():
    raise FileNotFoundError(
        f"Не найден отчёт rollout-оценки: {rollout_eval_path}. "
        "Сначала запустите GNN.evaluate_rollout для baseline, BPTT-3 и BPTT-5."
    )

rollout_eval_df = pd.DataFrame(json.loads(rollout_eval_path.read_text(encoding="utf-8")))
model_order = [model for model in ["baseline", "bptt3", "bptt5"] if model in rollout_eval_df["model"].unique()]
model_labels = {"baseline": "Baseline", "bptt3": "BPTT-3", "bptt5": "BPTT-5"}
model_colors = {"baseline": "#444444", "bptt3": "#1f77b4", "bptt5": "#d62728"}

metric_specs = [
    ("position_mean", "Средняя ошибка позиции", "position mean, raw units"),
    ("position_rmse", "RMSE позиции", "position RMSE, raw units"),
    ("shape_rmse", "RMSE формы", "shape RMSE"),
    ("valid_shape_fraction", "Доля валидных форм", "valid shapes, %"),
]

fig_rollout_metrics = make_subplots(
    rows=2, cols=2,
    subplot_titles=[title for _, title, _ in metric_specs],
    horizontal_spacing=0.11,
    vertical_spacing=0.16,
)

for metric_index, (metric, _, y_title) in enumerate(metric_specs):
    row = metric_index // 2 + 1
    col = metric_index % 2 + 1
    for model in model_order:
        model_df = rollout_eval_df[rollout_eval_df["model"] == model].sort_values("horizon")
        values = model_df[metric] * 100.0 if metric == "valid_shape_fraction" else model_df[metric]
        fig_rollout_metrics.add_trace(
            go.Scatter(
                x=model_df["horizon"],
                y=values,
                mode="lines+markers",
                name=model_labels[model],
                legendgroup=model,
                showlegend=(metric_index == 0),
                line=dict(color=model_colors[model], width=3),
                marker=dict(color=model_colors[model], size=7),
                hovertemplate=(
                    f"{model_labels[model]}<br>шаг=%{{x}}<br>{y_title}=%{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=col,
        )
    fig_rollout_metrics.update_xaxes(title_text="rollout step", tickmode="array", tickvals=[1, 3, 5, 10, 20], row=row, col=col)
    fig_rollout_metrics.update_yaxes(title_text=y_title, rangemode="tozero", row=row, col=col)

fig_rollout_metrics.update_layout(
    title="Test rollout: качество позиции и формы по горизонту",
    width=1100,
    height=760,
    plot_bgcolor="white",
    hovermode="x unified",
    legend=dict(
        x=1.01, y=1, xanchor="left",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
        groupclick="togglegroup",
    ),
    margin=dict(l=70, r=170, t=90, b=60),
)
fig_rollout_metrics.update_xaxes(showgrid=True, gridcolor="#e5e5e5", zeroline=False)
fig_rollout_metrics.update_yaxes(showgrid=True, gridcolor="#e5e5e5", zeroline=False)
fig_rollout_metrics.show()

h20_summary = (
    rollout_eval_df[rollout_eval_df["horizon"] == 20]
    .set_index("model")
    .reindex(model_order)
    .reset_index()
    .assign(
        model=lambda df: df["model"].map(model_labels),
        valid_shape_percent=lambda df: 100.0 * df["valid_shape_fraction"],
    )[["model", "matched_nodes", "position_mean", "position_median", "position_rmse", "shape_rmse", "valid_shape_percent"]]
    .rename(columns={
        "model": "Модель",
        "matched_nodes": "GT клеток",
        "position_mean": "Position mean",
        "position_median": "Position median",
        "position_rmse": "Position RMSE",
        "shape_rmse": "Shape RMSE",
        "valid_shape_percent": "Валидные формы, %",
    })
)

display(h20_summary.style.format({
    "Position mean": "{:.4f}",
    "Position median": "{:.4f}",
    "Position RMSE": "{:.4f}",
    "Shape RMSE": "{:.4f}",
    "Валидные формы, %": "{:.2f}",
}))

print("Вывод: BPTT-3 — основной checkpoint для траекторий и стабильных форм на длинном rollout.")
print("BPTT-5 даёт минимальный Shape RMSE, но на шаге 20 уступает BPTT-3 по средней ошибке позиции и доле валидных форм.")


,Модель,GT клеток,Position mean,Position median,Position RMSE,Shape RMSE,"Валидные формы, %"
0,Baseline,3954,5.8416,5.0006,7.2036,0.6075,11.61
1,BPTT-3,3954,5.6255,4.7088,6.9160,0.3270,95.73
2,BPTT-5,3954,5.6424,4.8030,6.8981,0.2832,84.42


Вывод: BPTT-3 — основной checkpoint для траекторий и стабильных форм на длинном rollout.
BPTT-5 даёт минимальный Shape RMSE, но на шаге 20 уступает BPTT-3 по средней ошибке позиции и доле валидных форм.


### Что произошло во время rollout-обучения

Train/validation loss для BPTT-3 и BPTT-5 нельзя сравнивать напрямую по абсолютному значению: BPTT-5 суммирует ошибку на более длинном rollout-окне. Сравнивать нужно форму кривых и итоговые test rollout-метрики выше.

- Для **BPTT-3** лучший checkpoint получен на эпохе 11: validation loss продолжал улучшаться достаточно долго.
- Для **BPTT-5** лучший checkpoint получен уже на эпохе 1. Train loss снижался, но validation loss затем ухудшался — это признак того, что текущая настройка BPTT-5 сложнее оптимизируется и начинает переобучаться.
- Текущий дифференцируемый rollout обучает движение и форму фиксированного набора клеток. Рождения, деления и смерти клеток в этом сравнении не моделируются.


In [33]:
rollout_run_dirs = {
    "BPTT-3": PROJECT_ROOT / "GNN" / "runs" / "gnn_rollout_bptt3_full_seed17",
    "BPTT-5": PROJECT_ROOT / "GNN" / "runs" / "gnn_rollout_bptt5_full_seed17",
}

training_rows = []
training_summaries = []
for label, run_dir in rollout_run_dirs.items():
    history_path = run_dir / "history.json"
    summary_path = run_dir / "run_summary.json"
    if not history_path.exists() or not summary_path.exists():
        print(f"{label}: нет history.json или run_summary.json, пропуск")
        continue
    history = json.loads(history_path.read_text(encoding="utf-8"))
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    for row in history:
        training_rows.append({"model": label, **row})
    training_summaries.append({
        "Модель": label,
        "Rollout steps": summary["config"]["rollout_steps"],
        "Лучшая эпоха": summary["best_epoch"],
        "Лучший validation loss": summary["best_metric"],
        "Checkpoint": str(run_dir / "best.pt"),
    })

training_history_df = pd.DataFrame(training_rows)
training_summary_df = pd.DataFrame(training_summaries)

fig_training = make_subplots(
    rows=1, cols=2,
    subplot_titles=list(rollout_run_dirs.keys()),
    shared_yaxes=False,
    horizontal_spacing=0.12,
)

for col, label in enumerate(rollout_run_dirs.keys(), start=1):
    model_history = training_history_df[training_history_df["model"] == label]
    if model_history.empty:
        continue
    for phase, color, dash in [("train", "#555555", "solid"), ("val", "#1f77b4", "dash")]:
        phase_df = model_history[model_history["phase"] == phase].sort_values("epoch")
        fig_training.add_trace(
            go.Scatter(
                x=phase_df["epoch"], y=phase_df["loss_total"],
                mode="lines+markers",
                name=phase,
                legendgroup=phase,
                showlegend=(col == 1),
                line=dict(color=color, width=2.5, dash=dash),
                marker=dict(color=color, size=5),
                hovertemplate=f"{label} | {phase}<br>epoch=%{{x}}<br>loss=%{{y:.4f}}<extra></extra>",
            ),
            row=1, col=col,
        )
    best_row = training_summary_df[training_summary_df["Модель"] == label].iloc[0]
    fig_training.add_vline(
        x=float(best_row["Лучшая эпоха"]),
        line_width=1.8,
        line_dash="dot",
        line_color="#d62728",
        annotation_text=f"best epoch {int(best_row['Лучшая эпоха'])}",
        annotation_position="top right",
        row=1, col=col,
    )
    fig_training.update_xaxes(title_text="epoch", dtick=1, row=1, col=col)
    fig_training.update_yaxes(title_text="total loss", row=1, col=col)

fig_training.update_layout(
    title="Rollout training: train и validation loss",
    width=1100,
    height=460,
    plot_bgcolor="white",
    legend=dict(x=1.01, y=1, xanchor="left"),
    margin=dict(l=70, r=130, t=90, b=60),
)
fig_training.update_xaxes(showgrid=True, gridcolor="#e5e5e5", zeroline=False)
fig_training.update_yaxes(showgrid=True, gridcolor="#e5e5e5", zeroline=False)
fig_training.show()

display(training_summary_df.style.format({"Лучший validation loss": "{:.6f}"}))


,Модель,Rollout steps,Лучшая эпоха,Лучший validation loss,Checkpoint
0,BPTT-3,3,11,1.728097,D:\Proga\Game_of_life\Real_game_of_life\GNN\runs\gnn_rollout_bptt3_full_seed17\best.pt
1,BPTT-5,5,1,2.786086,D:\Proga\Game_of_life\Real_game_of_life\GNN\runs\gnn_rollout_bptt5_full_seed17\best.pt


## Какие данные передаются в GNN для предсказания

Для каждого кадра GNN получает один граф. Ниже печатается фактический список входов для текущего `graph`: признаки узлов, признаки рёбер и сама структура связей. Целевые значения следующего шага сюда не входят: они используются только при обучении как ответы для сравнения.

In [34]:
if "graph" not in globals() or "nodes" not in globals():
    raise RuntimeError("Сначала выполните ячейку, которая строит GNN-граф для выбранного кадра.")

focus_spot_id = globals().get("center_spot_id", 6972)
node_ids_numeric = pd.to_numeric(nodes[gnn_cfg.node_id_col], errors="coerce")
focus_matches = np.flatnonzero(node_ids_numeric.to_numpy() == focus_spot_id)
if len(focus_matches) == 0:
    focus_node_index = 0
    focus_spot_id = nodes.loc[focus_node_index, gnn_cfg.node_id_col]
    print(f"spot_id=6972 не найден в текущем графе, показываю первый узел: {focus_spot_id}")
else:
    focus_node_index = int(focus_matches[0])

print("1. Узлы графа")
print(f"   - один узел = одна клетка/spot в текущем кадре")
print(f"   - число узлов: {graph.num_nodes}")
print(f"   - id узла: {gnn_cfg.node_id_col}")
print(f"   - значения ниже показаны для spot_id={focus_spot_id}")
print()

print("2. Признаки выбранного узла, которые попадают в data.x")
node_feature_values = pd.Series(
    graph.x[focus_node_index],
    index=list(graph.node_feature_columns),
    name=f"spot_id={focus_spot_id}",
)
for index, (column, value) in enumerate(node_feature_values.items(), start=1):
    print(f"   {index:02d}. {column} = {value:.6g}")
print(f"   Всего node features: {len(graph.node_feature_columns)}")
print()

display(
    node_feature_values
    .rename_axis("node_feature")
    .reset_index(name="value")
)
print()

print("3. Рёбра графа, которые попадают в data.edge_index")
print("   - ребро = связь между двумя клетками в текущем кадре")
print(f"   - число рёбер: {graph.num_edges}")
print(f"   - правило построения: radius={gnn_cfg.edge_radius}, k_nearest={gnn_cfg.edge_k_nearest}, bidirectional={gnn_cfg.bidirectional_edges}")
print()

print("4. Признаки рёбер, которые попадают в data.edge_attr")
for index, column in enumerate(graph.edge_feature_columns, start=1):
    print(f"   {index:02d}. {column}")
print(f"   Всего edge features: {len(graph.edge_feature_columns)}")
print()

edge_rows = []
for edge_position, (source_index, target_index) in enumerate(graph.edge_index.T):
    source_index = int(source_index)
    target_index = int(target_index)
    if source_index != focus_node_index and target_index != focus_node_index:
        continue
    row = {
        "edge_position": edge_position,
        "source_spot_id": nodes.loc[source_index, gnn_cfg.node_id_col],
        "target_spot_id": nodes.loc[target_index, gnn_cfg.node_id_col],
    }
    for feature_index, column in enumerate(graph.edge_feature_columns):
        row[column] = graph.edge_attr[edge_position, feature_index]
    edge_rows.append(row)

if edge_rows:
    focus_edges = pd.DataFrame(edge_rows)
    print(f"   Значения edge features для рёбер, связанных с spot_id={focus_spot_id}:")
    display(focus_edges)
else:
    print(f"   У spot_id={focus_spot_id} нет рёбер в текущем графе.")
print()

print("5. Что НЕ передаём как вход для предсказания")
excluded_prefixes = ("target_", "division_within_", "eligible_within_")
excluded_examples = [column for column in nodes.columns if column.startswith(excluded_prefixes)]
print("   - target_* признаки следующего шага")
print("   - division_within_* и eligible_within_* метки горизонтов")
print("   - next_id / prev_id и другие служебные ссылки треков")
print(f"   - найдено служебных/target колонок в таблице: {len(excluded_examples)}")
print()

print("6. Что модель предсказывает по этим входам")
print("   - delta_pos: сдвиг клетки к следующему кадру")
print("   - delta_shape: изменение формы")
print("   - division_logits: вероятность деления")
print("   - death_logits: вероятность исчезновения/смерти")
print("   - division_horizon_logits: вероятность деления на заданных горизонтах, если эта голова включена")

1. Узлы графа
   - один узел = одна клетка/spot в текущем кадре
   - число узлов: 22
   - id узла: spot_id
   - значения ниже показаны для spot_id=6972

2. Признаки выбранного узла, которые попадают в data.x
   01. x = 239.57
   02. y = 283.045
   03. AREA = 798.5
   04. PERIMETER = 107.045
   05. CIRCULARITY = 0.875701
   06. SOLIDITY = 0.982165
   07. RADIUS = 15.9427
   08. ELLIPSE_MAJOR = 19.9415
   09. ELLIPSE_MINOR = 12.9412
   10. ELLIPSE_ASPECTRATIO = 1.54094
   11. ELLIPSE_THETA = -0.80226
   12. MEAN_INTENSITY_CH1 = 3257.95
   13. contour_point_count = 32
   14. shape_missing_fraction = 0
   15. shape_area_ratio = 1
   16. shape_reconstruction_area_ratio = 1.00042
   17. shape_contour_centroid_dx = 2.11986e-05
   18. shape_contour_centroid_dy = -1.49562e-05
   19. shape_mean_radius = 15.7635
   20. shape_radius_std = 2.40581
   21. shape_radius_cv = 0.152619
   22. n_neighbors = 0
   23. density = 0
   24. Fx = 0
   25. Fy = 0
   26. shape_r_norm_000 = 1.26114
   27. shape_r_

,node_feature,value
0,x,239.569916
1,y,283.045197
2,AREA,798.500000
3,PERIMETER,107.044548
4,CIRCULARITY,0.875701
...,...,...
84,shape_r_norm_059,1.079122
85,shape_r_norm_060,1.115663
86,shape_r_norm_061,1.166392
87,shape_r_norm_062,1.158484



3. Рёбра графа, которые попадают в data.edge_index
   - ребро = связь между двумя клетками в текущем кадре
   - число рёбер: 42
   - правило построения: radius=80.0, k_nearest=0, bidirectional=True

4. Признаки рёбер, которые попадают в data.edge_attr
   01. dx
   02. dy
   03. distance
   04. unit_dx
   05. unit_dy
   Всего edge features: 5

   Значения edge features для рёбер, связанных с spot_id=6972:


,edge_position,source_spot_id,target_spot_id,dx,dy,distance,unit_dx,unit_dy
0,8,6984,6972,0.299495,-43.040089,43.041130,0.006958,-0.999976
1,12,6954,6972,66.657211,22.626429,70.392746,0.946933,0.321431
2,15,6986,6972,-53.276646,-51.208904,73.896904,-0.720959,-0.692978
3,32,6972,6984,-0.299495,43.040089,43.041130,-0.006958,0.999976
4,33,6972,6954,-66.657211,-22.626429,70.392746,-0.946933,-0.321431
5,34,6972,6986,53.276646,51.208904,73.896904,0.720959,0.692978
6,35,6972,6941,38.250420,-62.424549,73.211472,0.522465,-0.852661
7,38,6941,6972,-38.250420,62.424549,73.211472,-0.522465,0.852661



5. Что НЕ передаём как вход для предсказания
   - target_* признаки следующего шага
   - division_within_* и eligible_within_* метки горизонтов
   - next_id / prev_id и другие служебные ссылки треков
   - найдено служебных/target колонок в таблице: 6

6. Что модель предсказывает по этим входам
   - delta_pos: сдвиг клетки к следующему кадру
   - delta_shape: изменение формы
   - division_logits: вероятность деления
   - death_logits: вероятность исчезновения/смерти
   - division_horizon_logits: вероятность деления на заданных горизонтах, если эта голова включена
